# Explore Performance For A Page Set

Compare weekly performance for a hand-picked set of pages.

## What this notebook helps answer
- How are a small set of pages trending week by week?
- Are clicks and impressions moving together, or differently?
- Are engaged sessions following the same pattern as search visibility?

## How to use
1. Run **Setup (run once)**.
2. In **Parameters**, paste `1-10` page paths separated by commas.
3. Run the remaining cells from top to bottom.

This notebook is intentionally simple: it focuses on three weekly trend charts for the pages you choose.

In [1]:
#@title Setup (run once)
import os
import sys
from datetime import date, timedelta

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import get_client, load_sql_template, run_query

lifeline_theme.inject_fonts()
client = get_client()

In [2]:
#@title Parameters
RECENT_WEEKS = 8 #@param {type:"integer"}
SELECTED_PAGE_PATHS = "/chat, /get-help/support-toolkit/topics/self-harm" #@param {type:"string"}


def parse_page_paths(raw_value: str) -> list[str]:
    page_paths: list[str] = []
    seen: set[str] = set()

    for value in raw_value.replace("\n", ",").split(","):
        page_path = value.strip()
        if page_path and page_path not in seen:
            page_paths.append(page_path)
            seen.add(page_path)

    return page_paths


if RECENT_WEEKS < 2:
    raise ValueError("RECENT_WEEKS must be at least 2.")

selected_page_paths = parse_page_paths(SELECTED_PAGE_PATHS)

if not selected_page_paths:
    raise ValueError("Add at least one page path in SELECTED_PAGE_PATHS.")

if len(selected_page_paths) > 10:
    raise ValueError("Pick no more than 10 page paths so the charts stay readable.")

analysis_end_date = date.today() - timedelta(days=2)

print(f"Project: {config.PROJECT_ID}")
print(f"Search Console dataset: {config.SEARCHCONSOLE_DATASET}")
print(f"GA4 dataset: {config.GA4_DATASET}")
print(f"Selected pages ({len(selected_page_paths)}):")
for page_path in selected_page_paths:
    print(f"- {page_path}")

Project: lifeline-website-480522
Search Console dataset: searchconsole
GA4 dataset: analytics_315584957
Selected pages (2):
- /chat
- /get-help/support-toolkit/topics/self-harm


In [3]:
# Freshness check and chart window
freshness_sql = load_sql_template(
    "sql/notebooks/seo_search_freshness.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
    ga4_dataset=config.GA4_DATASET,
)

freshness = run_query(client, freshness_sql)
gsc_max_date = pd.to_datetime(freshness.loc[0, "gsc_max_date"]).date()
ga4_max_date = pd.to_datetime(freshness.loc[0, "ga4_max_date"]).date()
seo_max_date = pd.to_datetime(freshness.loc[0, "seo_max_date"]).date()

effective_end_date = min(analysis_end_date, gsc_max_date, ga4_max_date, seo_max_date)
latest_week_start = pd.Timestamp(effective_end_date).to_period("W-SUN").start_time
recent_week_cutoff = latest_week_start - pd.Timedelta(days=(RECENT_WEEKS - 1) * 7)
analysis_start_date = recent_week_cutoff.date()
recent_weeks = pd.date_range(
    start=recent_week_cutoff.normalize(),
    end=latest_week_start.normalize(),
    freq="W-MON",
)

print(f"Latest GSC date: {gsc_max_date}")
print(f"Latest GA4 date: {ga4_max_date}")
print(f"Latest joined SEO page date: {seo_max_date}")
print(f"Effective analysis end date: {effective_end_date}")
print(f"Chart week range: {recent_weeks.min().date()} to {recent_weeks.max().date()}")
print(f"Weeks included: {len(recent_weeks)}")

Latest GSC date: 2026-03-07
Latest GA4 date: 2026-03-08
Latest joined SEO page date: 2026-03-08
Effective analysis end date: 2026-03-07
Chart week range: 2026-01-12 to 2026-03-02
Weeks included: 8


In [4]:
# Load weekly metrics for the selected page set
page_sql = load_sql_template(
    "sql/notebooks/seo_page_set_weekly_performance.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

page_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
    bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    bigquery.ArrayQueryParameter("selected_page_paths", "STRING", selected_page_paths),
]

df_page_set_weekly = run_query(client, page_sql, params=page_params)

if df_page_set_weekly.empty:
    raise ValueError("No weekly page data returned for the selected pages.")

df_page_set_weekly["week_start"] = pd.to_datetime(df_page_set_weekly["week_start"])
df_page_set_weekly = df_page_set_weekly.sort_values(["page_order", "week_start"]).reset_index(drop=True)

selected_page_summary = (
    df_page_set_weekly.groupby("page_path", sort=False, as_index=False)
    .agg(
        clicks=("clicks", "sum"),
        impressions=("impressions", "sum"),
        engaged_sessions=("engaged_sessions", "sum"),
    )
)

zero_metric_pages = selected_page_summary[
    (selected_page_summary["clicks"] == 0)
    & (selected_page_summary["impressions"] == 0)
    & (selected_page_summary["engaged_sessions"] == 0)
]["page_path"].tolist()

print(f"Rows returned: {len(df_page_set_weekly):,}")
if zero_metric_pages:
    print("Warning: these selected pages returned all-zero metrics in this window:")
    for page_path in zero_metric_pages:
        print(f"- {page_path}")

selected_page_summary

Rows returned: 16


,page_path,clicks,impressions,engaged_sessions
0,/chat,5135,1067273,17358
1,/get-help/support-toolkit/topics/self-harm,4033,171963,1334


## Weekly charts for the selected pages

These charts mirror the first chart style from notebook `08`, but they use your chosen page set instead of automatically picked top movers.

How to read them:
- Each line is one selected page.
- The x-axis is the start of each week.
- The third chart uses GA4 `engaged_sessions`, so it helps show whether visits stayed meaningfully active after arriving.

In [5]:
# Three weekly trend charts in the same visual style
ndf = df_page_set_weekly.copy()

date_label = f"{recent_weeks.min().strftime('%d %b %Y')} to {recent_weeks.max().strftime('%d %b %Y')}"
selected_label = f"{len(selected_page_paths)} selected page(s)"


def show_metric_chart(metric_col: str, metric_title: str, y_axis_label: str) -> None:
    chart_title = (
        f"{metric_title}"
        f"<br><span style='font-size:0.8em; font-weight:400;'>{date_label}</span>"
        f"<br><span style='font-size:0.55em; font-weight:400;'>{selected_label}</span>"
    )

    fig = px.line(
        ndf,
        x="week_start",
        y=metric_col,
        color="page_path",
        category_orders={"page_path": selected_page_paths},
        markers=True,
        template="lifeline",
        title=chart_title,
        labels={
            "page_path": "Page path",
            "week_start": "Week",
            metric_col: y_axis_label,
        },
    )
    fig.update_xaxes(range=[recent_weeks.min(), recent_weeks.max()])
    fig.update_layout(legend_title_text="Page path")
    lifeline_theme.add_lifeline_logo(fig)
    fig.show()


show_metric_chart("clicks", "Weekly clicks from search", "Clicks")
show_metric_chart("impressions", "Weekly impressions from search", "Impressions")
show_metric_chart("engaged_sessions", "Weekly engaged sessions from GA4", "Engaged sessions")